In [ ]:
import pandas as pd
from pathlib import Path
df = pd.read_parquet('predictions_01_11_2022.parquet')
df = df.fillna('')
df

In [ ]:
# Reorder columns
order = ['comp', 'Tg', 'Tm',  'YM', 'TS_b',  'eps_b', 'perm_O2', 'perm_CO2', 'Td', 'TS_y', 'perm_N2',  
       'perm_H2',  'perm_He',   'perm_CH4', 'smiles', 'num_side', 'num_back', 'end_group', 'names']
df = df[order]
df

In [ ]:
df_exp = pd.read_pickle('exp_values.pkl')

# PLA properties from published literature
# Each value traced to a specific peer-reviewed source
pla_row = pd.DataFrame({
    'Polymer': ['Poly(lactic acid)'],
    'Abb.': ['PLA'],
    'applications': ['Packaging, 3D printing, medical implants'],
    'smiles': ['[*]OC(C)C(=O)[*]'],
    '%-usage': [0],
    'Tg': [333],       # Avérous 2008: reports 323-353 K range; 333 K is midpoint for typical commercial PLA
    'Tm': [443],        # Farah et al. 2016, Adv. Drug Deliv. Rev.: Table of physical properties of high Mw PLA
    'TS_b': [60.0],     # Santos et al. 2015, Procedia Eng. 114: 62.7 MPa for neat PLA (NatureWorks, 170k g/mol); rounded to 60
    'eps_b': [6.0],     # MakeItFrom.com (citing Auras et al. 2010, PLA textbook); also Wikipedia: "less than 10%", 6% for pure PLLA
    'YM': [1900],       # Farah et al. 2016, Adv. Drug Deliv. Rev.: comparison table of PLA physical properties
    'perm_O2': [0.26],  # Bao, Dorgan et al. 2006, J. Membr. Sci. 285, 166-172: time-lag method, PLA 98.7% L at 30°C
    'perm_CO2': [1.10],  # Bao, Dorgan et al. 2006, J. Membr. Sci. 285, 166-172: time-lag method, PLA 98.7% L at 30°C
    'source': ['Tg: Avérous 2008; Tm: Farah et al. 2016; TS_b: Santos et al. 2015; eps_b: Auras et al. 2010; YM: Farah et al. 2016; perm_O2/CO2: Bao & Dorgan 2006']
})

df_exp = pd.concat([df_exp, pla_row], ignore_index=True)
df_exp

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# props to match
search_props = ['Tg', 'Tm', 'TS_b', 'YM',  'eps_b', 'perm_O2', 'perm_CO2']

# weights 
weights = np.array([1, 1, 1, 1, 1, 1, 1])
params = dict(
    metric='minkowski',
    p=2
)

# PHAs only
df_pha_only = df[df.names.apply(lambda x: list(x) == ['pha', 'pha'])]
nn_pha = NearestNeighbors(**params).fit(df_pha_only[search_props])

# All others (with conventional polymers)
df_pha_conv = df[df.names.apply(lambda x: list(x) != ['pha', 'pha'])]
nn_conv = NearestNeighbors(**params).fit(df_pha_conv[search_props])


# Nearest Neighbor for phas phas

In [ ]:

n_neighbors = 5 
results = []
for n, row in df_exp.iterrows():
    r = row[search_props]
    dist, idx = nn_pha.kneighbors(r.to_frame().T, n_neighbors)
    res = df_pha_only.iloc[idx[0]].copy()
    res.insert(0, 'polymer', row.Polymer)
    res.insert(1, 'abb', row['Abb.'])
    res.insert(1, 'type', 'pha_only')

    diff = res[search_props] - r.to_frame().T.values
    diff = diff.add_prefix('d_')
    res = pd.concat([res, diff], axis=1)
    results.append(res)

    dist, idx = nn_conv.kneighbors(r.to_frame().T, n_neighbors)
    res = df_pha_conv.iloc[idx[0]].copy()
    res.insert(0, 'polymer', row.Polymer)
    res.insert(1, 'abb', row['Abb.'])
    res.insert(2, 'type', 'pha_conv')

    diff = res[search_props] - r.to_frame().T.values
    diff = diff.add_prefix('d_')
    res = pd.concat([res, diff], axis=1)

    results.append(res)

df_res = pd.concat(results).set_index(['polymer', 'type'])  

df_res.insert(0, 'smiles', df_res.pop('smiles'))
df_res.insert(0, 'names', df_res.pop('names'))

pp = Path('export')
pp.mkdir(exist_ok=True)
df_res.to_csv(pp / 'candidates.csv')
df_res[df_res.abb == 'PLA']

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

import numpy as np

for name, gr in df_res.reset_index().groupby(by="abb", sort=False):
    sms = np.array([[x[0], x[1]] for x in gr.smiles.values])
    mols = [Chem.MolFromSmiles(x) for x in sms.flatten()]
    comps = gr.comp.values

    for num, sm in enumerate(sms):
        sms[num][0] = f"{round(comps[num],2)}: {sms[num][0]}"
    nn = name.lower().replace(" ", "")
    img = Draw.MolsToGridImage(
        mols,
        subImgSize=(200, 200),
        molsPerRow=2,
        maxMols=200,
        useSVG=True,
        legends=sms.flatten().tolist(),
    )
    Path(pp / f"{nn}.svg").write_text(img.data)


## Aliphatic PHA Screening

Re-run the nearest-neighbor search after removing all candidates where
any PHA monomer contains aromatic atoms. This produces bio-replacements
using only aliphatic (non-aromatic) PHAs, which are more realistic for
biological production and biodegradation.

In [ ]:
from utils.chemistry import has_aromatic_atoms

def row_has_aromatic_pha(row):
    """Check if any PHA monomer in this row contains aromatic atoms."""
    for name, smi in zip(row['names'], row['smiles']):
        if name == 'pha' and has_aromatic_atoms(smi):
            return True
    return False

n_before = len(df)
df_aliphatic = df[~df.apply(row_has_aromatic_pha, axis=1)]
n_after = len(df_aliphatic)

print(f"Filtered aromatic PHA monomers: {n_before:,} → {n_after:,} rows ({n_before - n_after:,} removed)")

In [ ]:
df_pha_only_ali = df_aliphatic[df_aliphatic.names.apply(lambda x: tuple(x) == ('pha', 'pha'))]
nn_pha_ali = NearestNeighbors(**params).fit(df_pha_only_ali[search_props])

df_pha_conv_ali = df_aliphatic[df_aliphatic.names.apply(lambda x: tuple(x) != ('pha', 'pha'))]
nn_conv_ali = NearestNeighbors(**params).fit(df_pha_conv_ali[search_props])

print(f"PHA-only candidates:         {len(df_pha_only_ali):,}")
print(f"PHA-conventional candidates: {len(df_pha_conv_ali):,}")

In [ ]:
results_ali = []
for n, row in df_exp.iterrows():
    r = row[search_props]

    dist, idx = nn_pha_ali.kneighbors(r.to_frame().T, n_neighbors)
    res = df_pha_only_ali.iloc[idx[0]].copy()
    res.insert(0, 'polymer', row.Polymer)
    res.insert(1, 'abb', row['Abb.'])
    res.insert(1, 'type', 'pha_only')
    diff = res[search_props] - r.to_frame().T.values
    diff = diff.add_prefix('d_')
    res = pd.concat([res, diff], axis=1)
    results_ali.append(res)

    dist, idx = nn_conv_ali.kneighbors(r.to_frame().T, n_neighbors)
    res = df_pha_conv_ali.iloc[idx[0]].copy()
    res.insert(0, 'polymer', row.Polymer)
    res.insert(1, 'abb', row['Abb.'])
    res.insert(2, 'type', 'pha_conv')
    diff = res[search_props] - r.to_frame().T.values
    diff = diff.add_prefix('d_')
    res = pd.concat([res, diff], axis=1)
    results_ali.append(res)

df_res_ali = pd.concat(results_ali).set_index(['polymer', 'type'])
df_res_ali.insert(0, 'smiles', df_res_ali.pop('smiles'))
df_res_ali.insert(0, 'names', df_res_ali.pop('names'))

df_res_ali.to_csv(pp / 'candidates_aliphatic.csv')
print(f"Saved {len(df_res_ali)} aliphatic candidates to export/candidates_aliphatic.csv")
df_res_ali

In [ ]:
# --- Visualization: Predicted vs Experimental Properties ---

import matplotlib.pyplot as plt

# Load candidates
df_orig = pd.read_csv(pp / 'candidates.csv')
df_ali = pd.read_csv(pp / 'candidates_aliphatic.csv')

# Get top-1 pha_only candidate per plastic
top1_orig = df_orig[df_orig['type'] == 'pha_only'].groupby('abb').first().reset_index()
top1_ali = df_ali[df_ali['type'] == 'pha_only'].groupby('abb').first().reset_index()

# Properties to plot and their display labels
props = ['Tg', 'Tm', 'TS_b', 'YM', 'eps_b', 'perm_O2', 'perm_CO2']
labels = [r'$T_g$ [K]', r'$T_m$ [K]', r'$\sigma_b$ [MPa]', r'$E$ [MPa]',
          r'$\epsilon_b$', r'$\mu_{O_2}$ [barrer]', r'$\mu_{CO_2}$ [barrer]']

# Plastics in display order (matching Supp Fig 10)
plastics = ['PE', 'PP', 'PVC', 'PET', 'PS', 'Nylon 6', 'PEN']

# Build experiment values in the same order
exp_vals = {}
for p in plastics:
    row = df_exp[df_exp['Abb.'] == p].iloc[0]
    exp_vals[p] = row

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, (prop, label) in enumerate(zip(props, labels)):
    ax = axes[i]

    y_exp = [float(exp_vals[p][prop]) for p in plastics]

    y_orig = []
    y_ali = []
    for p in plastics:
        orig_row = top1_orig[top1_orig['abb'] == p]
        ali_row = top1_ali[top1_ali['abb'] == p]
        y_orig.append(float(orig_row[prop].values[0]) if len(orig_row) > 0 else None)
        y_ali.append(float(ali_row[prop].values[0]) if len(ali_row) > 0 else None)

    x = range(len(plastics))
    ax.plot(x, y_exp, 'o-', color='tab:red', label='Experiment', markersize=6)
    ax.plot(x, y_orig, 's--', color='tab:blue', label='Original (PHA-only)', markersize=6)
    ax.plot(x, y_ali, 'D:', color='tab:green', label='Aliphatic (PHA-only)', markersize=6)

    ax.set_ylabel(label)
    ax.set_xticks(x)
    ax.set_xticklabels(plastics, fontsize=8)
    if i == 0:
        ax.legend(fontsize=8)

# Hide unused subplots (we have 7 plots in a 3x3 grid)
for j in range(len(props), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Top-1 PHA-Only Candidates vs Experimental Properties', fontsize=13)
plt.tight_layout()
plt.savefig(pp / 'property_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- SVG structure visualization for aliphatic candidates ---

for name, gr in df_res_ali.reset_index().groupby(by="abb", sort=False):
    sms = np.array([[x[0], x[1]] for x in gr.smiles.values])
    mols = [Chem.MolFromSmiles(x) for x in sms.flatten()]
    comps = gr.comp.values

    for num, sm in enumerate(sms):
        sms[num][0] = f"{round(comps[num],2)}: {sms[num][0]}"
    nn = name.lower().replace(" ", "")
    img = Draw.MolsToGridImage(
        mols,
        subImgSize=(200, 200),
        molsPerRow=2,
        maxMols=200,
        useSVG=True,
        legends=sms.flatten().tolist(),
    )
    svg_text = img.data
    svg_text = svg_text.replace('<rect', '<rect fill="white" width="100%" height="100%"/><rect', 1)
    Path(pp / f"{nn}_aliphatic.svg").write_text(svg_text)

print(f"Saved aliphatic SVGs to {pp}/")

In [ ]:
# --- How far down the original rankings are the aliphatic candidates? ---

for n, row in df_exp.iterrows():
    plastic = row['Abb.']
    r = row[search_props]

    # Get distances from the ORIGINAL (unfiltered) NN model to ALL candidates
    dist_all, idx_all = nn_pha.kneighbors(r.to_frame().T, n_neighbors=len(df_pha_only))

    # Get the aliphatic top-1 for this plastic
    ali_row = df_res_ali.reset_index()
    ali_top1 = ali_row[(ali_row['abb'] == plastic) & (ali_row['type'] == 'pha_only')].iloc[0]

    # Find where this candidate sits in the original ranking
    ali_props = ali_top1[search_props].values.astype(float)
    original_props = df_pha_only.iloc[idx_all[0]][search_props].values

    # Distance of the original top-1 (for comparison)
    top1_dist = dist_all[0][0]

    for rank, orig_props in enumerate(original_props):
        if np.allclose(orig_props, ali_props, atol=1e-4):
            ali_dist = dist_all[0][rank]
            diff = ali_dist - top1_dist
            print(f"{plastic}: aliphatic top-1 was ranked #{rank + 1:,}  "
                  f"(distance: {ali_dist:.2f} vs top-1: {top1_dist:.2f}, "
                  f"Δ = +{diff:.2f})")
            break
    else:
        print(f"{plastic}: aliphatic top-1 not found in original ranking")

In [ ]:
# --- Distance vs Rank: where do aliphatic candidates fall? ---

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (n, row) in enumerate(df_exp.iterrows()):
    plastic = row['Abb.']
    r = row[search_props]
    ax = axes[i]

    # Get the aliphatic top-1 for this plastic
    ali_row = df_res_ali.reset_index()
    ali_top1 = ali_row[(ali_row['abb'] == plastic) & (ali_row['type'] == 'pha_only')].iloc[0]
    ali_props = ali_top1[search_props].values.astype(float)

    # Get enough ranked candidates to find the aliphatic one + buffer
    # Start with 500, increase if needed
    n_check = 500
    dist_chunk, idx_chunk = nn_pha.kneighbors(r.to_frame().T, n_neighbors=n_check)

    original_props = df_pha_only.iloc[idx_chunk[0]][search_props].values
    ali_rank = None
    for rank, orig_props in enumerate(original_props):
        if np.allclose(orig_props, ali_props, atol=1e-4):
            ali_rank = rank
            break

    # If not found in 500, expand search
    if ali_rank is None:
        n_check = len(df_pha_only)
        dist_chunk, idx_chunk = nn_pha.kneighbors(r.to_frame().T, n_neighbors=n_check)
        original_props = df_pha_only.iloc[idx_chunk[0]][search_props].values
        for rank, orig_props in enumerate(original_props):
            if np.allclose(orig_props, ali_props, atol=1e-4):
                ali_rank = rank
                break

    # Plot distance curve up to 2x the aliphatic rank (or at least 50)
    show_up_to = max(50, int(ali_rank * 2)) if ali_rank is not None else n_check
    show_up_to = min(show_up_to, len(dist_chunk[0]))

    ax.plot(range(1, show_up_to + 1), dist_chunk[0][:show_up_to],
            color='tab:blue', linewidth=1)

    if ali_rank is not None:
        ax.axvline(x=ali_rank + 1, color='tab:green', linestyle='--', linewidth=1.5,
                   label=f'Aliphatic top-1 (#{ali_rank + 1:,})')
        ax.plot(ali_rank + 1, dist_chunk[0][ali_rank], 'D',
                color='tab:green', markersize=8, zorder=5)

    ax.set_title(plastic, fontsize=11)
    ax.set_xlabel('Rank')
    ax.set_ylabel('Distance')
    ax.legend(fontsize=7)

# Hide unused subplot if fewer than 8 plastics
for j in range(len(df_exp), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Euclidean Distance vs Rank — Where Do Aliphatic Candidates Fall?', fontsize=13)
plt.tight_layout()
plt.savefig(pp / 'distance_vs_rank.png', dpi=150, bbox_inches='tight')
plt.show()